# ColorVision AI: A Computer Vision System for Single-Color and Multicolor Object Identification

**How to use this notebook:** run the cells **from top to bottom**. Each section starts with a
**📘 What & why** box that explains what you are about to do, so you can describe it in your own words in the report.
Cells marked **✍️ YOUR TURN** are where you write your own observations (the assessors want *your* explanations of why each change was made).

## 1. Introduction and objective

**Application.** Identify objects by *shape* and by *colour*, including objects made of several colours.

**Object classes (10):**

| Shape | Single-colour classes | Multicolour class |
|---|---|---|
| ball | `red_ball`, `blue_ball`, `green_ball`, `yellow_ball` | `multicolor_ball` |
| ring | `red_ring`, `blue_ring`, `green_ring`, `yellow_ring` | `multicolor_ring` |

**Objective.** Build two kinds of system and compare them:
1. **YOLO object detector**: learns shape *and* colour together from labelled boxes.
2. **Classical HSV colour analysis**: measures the colour directly from pixels (no learning), and reports the **percentage of each colour** inside an object.

A *hybrid* (YOLO finds the object, HSV measures its colour) is also tested.
The key lesson: **colour identification is not the same as object detection.**

> ⚠️ **Honesty note.** The dataset provided is *synthetic* (computer-generated shapes on plain backgrounds).
> Results will look very good, but they do **not** prove the system works on real photographs. State this in your conclusion.

✍️ **YOUR TURN:** write 2–3 sentences in your own words on the problem you are solving and why colour identification is useful (e.g. sorting, quality control, accessibility).

## 2. Environment setup

📘 **What & why**
- **GPU:** training a neural network is millions of small calculations. A GPU does them in parallel, ~10-50× faster than a CPU.
  Go to **Runtime → Change runtime type → T4 GPU** *before* running anything.
- **Ultralytics** is the library that provides YOLO (we use the `yolo11n` "nano" model: small and fast, ideal for free Colab).
- **Colab resets** when you disconnect, so the notebook saves results to Google Drive at the end.

In [ ]:
# Check that a GPU is attached (you should see "Tesla T4" or similar)
!nvidia-smi

In [ ]:
# Install YOLO (Ultralytics). -q = quiet output
!pip install -q ultralytics

In [ ]:
import os, glob, random, shutil, zipfile, time, yaml
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name     :", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU! Runtime → Change runtime type → T4 GPU, then re-run from the top.")

# Make results reproducible (same random numbers each run)
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 3. Dataset preparation

📘 **What & why**
- You will upload `ColorVision_AI_YOLO_300_images.zip` and unzip it.
- **YOLO format:** every image has a `.txt` label file; each line is one object: `class_id x_center y_center width height`
  (all numbers are *fractions* of the image size, between 0 and 1).
- **Splits:** `train` (210 images) is what the model learns from, `val` (60) is used to compare experiments,
  and `test` (30) is kept for the **final** check only, so we never "tune to the test".
- We then **inspect the class distribution**. If some classes have far fewer examples they are *weak classes* and the model usually does worse on them.

In [ ]:
# OPTION A (simplest): upload the zip from your computer. A file picker will appear.
from google.colab import files
uploaded = files.upload()          # choose ColorVision_AI_YOLO_300_images.zip

# OPTION B: if the zip is in your Google Drive, comment out the two lines above and use:
# from google.colab import drive; drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/ColorVision_AI_YOLO_300_images.zip" /content/

In [ ]:
# Unzip, find the dataset folder (the one containing data.yaml) and copy it to /content/colorvision_yolo_dataset
zip_path = glob.glob('/content/*.zip')[0]
os.makedirs('/content/raw', exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/raw')

src = os.path.dirname(glob.glob('/content/raw/**/data.yaml', recursive=True)[0])
DATA = '/content/colorvision_yolo_dataset'
if os.path.exists(DATA): shutil.rmtree(DATA)
shutil.copytree(src, DATA)

# Delete stale label caches (YOLO rebuilds them; old ones can cause confusing errors)
for c in glob.glob(f'{DATA}/labels/*.cache'): os.remove(c)

CLASS_NAMES = ['red_ball','blue_ball','green_ball','yellow_ball','multicolor_ball',
               'red_ring','blue_ring','green_ring','yellow_ring','multicolor_ring']
COLOR_OF = lambda name: name.split('_')[0]    # 'red_ball' -> 'red'
SHAPE_OF = lambda name: name.split('_')[1]    # 'red_ball' -> 'ball'

def make_yaml(root, filename='data_colab.yaml'):
    # Write a data.yaml with absolute paths so YOLO always finds the images.
    cfg = dict(path=root, train='images/train', val='images/val', test='images/test',
               names={i: n for i, n in enumerate(CLASS_NAMES)})
    out = f'{root}/{filename}'
    with open(out, 'w') as f: yaml.safe_dump(cfg, f, sort_keys=False)
    return out

DATA_YAML = make_yaml(DATA)
print(open(DATA_YAML).read())
for s in ['train','val','test']:
    print(s, len(glob.glob(f'{DATA}/images/{s}/*.jpg')), 'images')

In [ ]:
# --- Class distribution: how many labelled objects of each class are in each split? ---
def count_instances(root, split):
    c = Counter()
    for f in glob.glob(f'{root}/labels/{split}/*.txt'):
        for line in open(f):
            if line.strip():
                c[int(line.split()[0])] += 1
    return c

dist = pd.DataFrame({s: [count_instances(DATA, s)[i] for i in range(10)] for s in ['train','val','test']},
                    index=CLASS_NAMES)
dist['total'] = dist.sum(axis=1)
display(dist)

dist[['train','val','test']].plot(kind='bar', stacked=True, figsize=(10,4), title='Objects per class')
plt.ylabel('number of objects'); plt.tight_layout(); plt.show()

In [ ]:
# --- Show sample annotations: draw the ground-truth boxes on a few training images ---
def read_labels(lbl_path, w, h):
    # Returns a list of (class_id, x1, y1, x2, y2) in PIXELS (converted from YOLO's normalised format).
    out = []
    if not os.path.exists(lbl_path): return out
    for line in open(lbl_path):
        if not line.strip(): continue
        c, xc, yc, bw, bh = line.split()
        c = int(c); xc, yc, bw, bh = (float(xc)*w, float(yc)*h, float(bw)*w, float(bh)*h)
        x1, y1, x2, y2 = int(xc-bw/2), int(yc-bh/2), int(xc+bw/2), int(yc+bh/2)
        out.append((c, max(x1,0), max(y1,0), min(x2,w), min(y2,h)))
    return out

def label_path(img_path):
    return img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'

def draw_gt(img_path):
    img = cv2.imread(img_path); h, w = img.shape[:2]
    for c, x1, y1, x2, y2 in read_labels(label_path(img_path), w, h):
        cv2.rectangle(img, (x1,y1), (x2,y2), (0,160,0), 2)
        cv2.putText(img, CLASS_NAMES[c], (x1, max(y1-6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,120,0), 2)
    return img[..., ::-1]    # OpenCV uses BGR, matplotlib wants RGB

def show_grid(images, titles=None, cols=3, figsize=(15,4)):
    rows = (len(images)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(figsize[0], figsize[1]*rows))
    for ax in np.array(axes).ravel(): ax.axis('off')
    for ax, im, i in zip(np.array(axes).ravel(), images, range(len(images))):
        ax.imshow(im); ax.set_title(titles[i] if titles else '', fontsize=9)
    plt.tight_layout(); plt.show()

train_imgs = sorted(glob.glob(f'{DATA}/images/train/*.jpg'))
show_grid([draw_gt(p) for p in random.sample(train_imgs, 6)], cols=3)

✍️ **YOUR TURN:** look at the table and bar chart above. Which classes have the **fewest** examples? Write them down; they are your *weak classes* for Experiment 2.
Also note what the images look like (background, lighting). Why might a *synthetic* dataset be easier than real photos?

## 4. Baseline inference (Experiment 1: pretrained model)

📘 **What & why**
`yolo11n.pt` was trained by Ultralytics on **COCO** (80 everyday classes: person, car, "sports ball"...).
It has **never seen** "red_ring" or "multicolor_ball". Running it *before* custom training gives a **baseline**
so we can later prove that our training actually helped. We ask: *which required objects are missed or wrongly classified?*

`conf=0.25` means "only show detections the model is at least 25% sure about".

In [ ]:
test_imgs = sorted(glob.glob(f'{DATA}/images/test/*.jpg'))
base_model = YOLO('yolo11n.pt')                  # downloads the pretrained weights
base_res = base_model.predict(test_imgs, conf=0.25, verbose=False)

# What did the pretrained model "see"? (COCO names, not our classes)
seen = Counter(base_model.names[int(c)] for r in base_res for c in r.boxes.cls)
n_gt = sum(len(read_labels(label_path(p), 640, 480)) for p in test_imgs)
print(f"Real objects in the 30 test images : {n_gt}")
print(f"Detections by pretrained model      : {sum(seen.values())}")
print("Labels it used                      :", dict(seen))

base_infer_ms = float(np.mean([r.speed['inference'] for r in base_res]))
print(f"Average inference time              : {base_infer_ms:.1f} ms/image")

show_grid([r.plot()[..., ::-1] for r in base_res[:6]], cols=3)

✍️ **YOUR TURN (Experiment 1 findings):** what do you see? Typical observations: it may call balls a *"sports ball"* (right shape, but no colour info),
it never says *ring*, it never says *multicolor*, and it may miss many objects. Write which required objects were **missed** or **incorrectly classified** and why
(the model has no idea our 10 classes exist).

> The pretrained model's mAP on *our* 10 classes is **0 by definition**: none of its 80 COCO classes are our classes. We record that in the results table below.

## 5. Custom model training

📘 **What & why**
**Fine-tuning** = start from the pretrained weights (which already know edges, shapes, textures) and keep training on *your* data so it learns *your* 10 classes.
This needs far less data/time than training from zero.

Settings you must understand (and record):

| Setting | Meaning |
|---|---|
| `epochs` | How many full passes over the 210 training images. Too few → under-trained; too many → **overfitting** (memorises train set). |
| `imgsz` | Images are resized to this size before the network sees them. Bigger = sees small objects better but slower. |
| `batch` | How many images are processed together. Bigger = faster/smoother but needs more GPU memory. |
| `patience` | Stop early if validation score hasn't improved for this many epochs (a guard against overfitting). |

After training, **`best.pt`** is the weights from the epoch with the best validation score; we evaluate that one.

The helper function below trains one model, evaluates it on the **validation** split, and stores all metrics in `RESULTS`
so every experiment is recorded in the same format.

In [ ]:
RUNS = '/content/runs'
RESULTS   = {}   # experiment name -> overall metrics
PER_CLASS = {}   # experiment name -> per-class mAP50-95
SAVE_DIRS = {}   # experiment name -> folder with plots/weights
CONFIGS   = {}   # experiment name -> the settings used (for your report)

def summarize_metrics(m):
    # Turn an Ultralytics validation result into plain numbers.
    b = m.box
    per_class = {}
    for i, ci in enumerate(b.ap_class_index):
        per_class[CLASS_NAMES[int(ci)]] = dict(P=float(b.p[i]), R=float(b.r[i]),
                                               mAP50=float(b.ap50[i]), mAP50_95=float(b.ap[i]))
    overall = dict(P=float(b.mp), R=float(b.mr), mAP50=float(b.map50), mAP50_95=float(b.map),
                   infer_ms=float(m.speed['inference']))
    return overall, per_class

def train_and_eval(name, data_yaml, weights='yolo11n.pt', **train_args):
    # Train a YOLO model, evaluate its best weights on the VALIDATION split, store the results.
    args = dict(epochs=30, imgsz=640, batch=16, patience=15)   # defaults
    args.update(train_args)
    CONFIGS[name] = dict(weights=weights, data=os.path.basename(os.path.dirname(data_yaml)), **args)
    print(f'\n=== Training: {name} ===\n{CONFIGS[name]}')

    t0 = time.time()
    model = YOLO(weights)
    model.train(data=data_yaml, project=RUNS, name=name, exist_ok=True,
                seed=SEED, plots=True, verbose=False, **args)
    train_min = (time.time() - t0) / 60
    save_dir = Path(model.trainer.save_dir)

    best = YOLO(save_dir / 'weights' / 'best.pt')
    m = best.val(data=data_yaml, split='val', project=RUNS, name=name + '_val',
                 exist_ok=True, plots=True, verbose=False)
    overall, per_class = summarize_metrics(m)
    overall['train_min'] = round(train_min, 1)
    RESULTS[name], PER_CLASS[name], SAVE_DIRS[name] = overall, per_class, save_dir
    print({k: round(v, 3) for k, v in overall.items()})
    return best

def best_experiment(names):
    return max(names, key=lambda n: RESULTS[n]['mAP50_95'])

# Record Experiment 1 (pretrained, no custom training), see the note in section 4
RESULTS['E1 pretrained'] = dict(P=0.0, R=0.0, mAP50=0.0, mAP50_95=0.0, infer_ms=base_infer_ms, train_min=0)
PER_CLASS['E1 pretrained'] = {n: dict(P=0, R=0, mAP50=0, mAP50_95=0) for n in CLASS_NAMES}

In [ ]:
# ---- Custom baseline: fine-tune on the ORIGINAL dataset with default settings ----
# This takes a few minutes on a T4. You will see a table of loss values per epoch.
# (box_loss / cls_loss should go DOWN over time; that means the model is learning.)
custom_model = train_and_eval('custom_baseline', DATA_YAML, epochs=30, imgsz=640, batch=16)

✍️ **YOUR TURN:** record the configuration printed above (epochs, image size, batch size). Open the training curves
(next section) and note whether training loss and validation loss both fall.

## 6. Model evaluation

📘 **What & why: the required metrics**

| Metric | Meaning | In plain words |
|---|---|---|
| **Precision** | correct detections ÷ all detections | "When it says *red_ball*, is it really?" (few false alarms) |
| **Recall** | detected objects ÷ all real objects | "Did it find all the red balls?" (few misses) |
| **mAP@50** | average precision when a box counts as correct if IoU ≥ 0.50 | overall detection quality, lenient on box position |
| **mAP@50–95** | same, averaged over IoU 0.50 → 0.95 | overall quality, strict on box position (the hardest, most informative score) |
| **Inference time** | milliseconds to process one image | speed |

**IoU** (Intersection over Union) = overlap between predicted and true box ÷ their combined area. 1.0 = perfect overlap.

**Plots to understand**
- **Confusion matrix:** rows/columns are classes; off-diagonal cells show *which classes get mixed up*.
- **PR curve:** precision vs recall; a curve hugging the top-right corner is good.
- **results.png:** loss and metrics per epoch. If *train* loss keeps falling but *val* loss starts rising → **overfitting**.

In [ ]:
def show_results(name):
    # Table of metrics and the key plots for one experiment.
    print(f'===== {name} =====')
    print(pd.DataFrame([RESULTS[name]]).round(3).to_string(index=False))
    display(pd.DataFrame(PER_CLASS[name]).T.round(3))
    d = SAVE_DIRS[name]
    val_dir = Path(str(d) + '_val')
    figs = [(d/'results.png', 'Training curves (loss & metrics per epoch)'),
            (val_dir/'confusion_matrix_normalized.png', 'Confusion matrix (normalised)'),
            (val_dir/'BoxPR_curve.png', 'Precision-Recall curve')]
    for path, title in figs:
        if path.exists():
            plt.figure(figsize=(11,6)); plt.imshow(plt.imread(path)); plt.axis('off'); plt.title(title); plt.show()

show_results('custom_baseline')

✍️ **YOUR TURN:** from the per-class table, which class has the **lowest** mAP50-95 / recall? From the confusion matrix, which classes are mistaken for each other
(e.g. `multicolor_ball` ↔ `multicolor_ring`, or ball vs ring of the same colour)? These errors guide your experiments.

## 7. Performance improvement experiments

📘 **What & why: the scientific method**
A **controlled experiment changes ONE thing at a time**, so you know what caused any improvement.
Every experiment below is compared against the previous best using the same validation set.

| Exp | What changes | Question |
|---|---|---|
| 1 | Pretrained model (done in section 4) | Which objects are missed/misclassified? |
| 2 | **Dataset improvement**: add extra (augmented) examples of weak classes | Does the model now detect previously missed objects? |
| 3 | **Training optimisation**: more epochs; larger image size | Does it learn better without overfitting? |
| 4 | **Advanced**: colour-safe vs stronger augmentation | Does performance improve on unseen images? |

⚠️ **Colour is the label here!** YOLO's default augmentation randomly shifts hue (`hsv_h`). A big hue shift can turn a *red* ball *blue* while its label still says
`red_ball`, which teaches the model wrong information. That is why Experiment 4 tests turning hue-shift off.

### Experiment 2: Dataset improvement
We find the weak classes (fewest training objects) and create **extra training images** from the *training* images that contain them
(horizontal flip + small brightness/contrast change + slight noise, with box labels flipped to match).
Val and test sets are **untouched**, so the comparison stays fair (no leakage).
> 💡 Duplicated/augmented images are a weak substitute for **new real images**. If you can photograph and label real objects (e.g. with Roboflow or LabelImg) and add them to `train`, do it, and say so in your report.

In [ ]:
train_counts = count_instances(DATA, 'train')
mean_count = np.mean([train_counts[i] for i in range(10)])
weak_ids = [i for i in range(10) if train_counts[i] < 0.9 * mean_count]
print('Training objects per class:', {CLASS_NAMES[i]: train_counts[i] for i in range(10)})
print('Weak classes (below 90% of the mean):', [CLASS_NAMES[i] for i in weak_ids])

def build_augmented_dataset(src, dst, weak_ids, copies=3, seed=0):
    # Copy the dataset and add augmented copies of every TRAIN image that contains a weak class.
    rng = random.Random(seed); nrng = np.random.RandomState(seed)
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)
    for c in glob.glob(f'{dst}/labels/*.cache'): os.remove(c)
    added = 0
    for lf in sorted(glob.glob(f'{src}/labels/train/*.txt')):
        rows = [l.split() for l in open(lf) if l.strip()]
        if not any(int(r[0]) in weak_ids for r in rows): continue
        stem = os.path.splitext(os.path.basename(lf))[0]
        img = cv2.imread(f'{src}/images/train/{stem}.jpg')
        for k in range(copies):
            im2, rows2 = img.copy(), [r[:] for r in rows]
            if k % 2 == 0:                                   # horizontal flip
                im2 = im2[:, ::-1].copy()
                for r in rows2: r[1] = f'{1 - float(r[1]):.6f}'   # mirror x_center
            im2 = cv2.convertScaleAbs(im2, alpha=rng.uniform(0.8, 1.2), beta=rng.uniform(-20, 20))
            im2 = np.clip(im2 + nrng.normal(0, 4, im2.shape), 0, 255).astype(np.uint8)
            cv2.imwrite(f'{dst}/images/train/{stem}_aug{k}.jpg', im2)
            with open(f'{dst}/labels/train/{stem}_aug{k}.txt', 'w') as f:
                f.write('\n'.join(' '.join(r) for r in rows2) + '\n')
            added += 1
    print(f'Added {added} augmented training images.')

DATA2 = '/content/colorvision_dataset_v2'
build_augmented_dataset(DATA, DATA2, weak_ids, copies=3)
DATA2_YAML = make_yaml(DATA2)

new_counts = count_instances(DATA2, 'train')
cmp = pd.DataFrame({'before': [train_counts[i] for i in range(10)],
                    'after':  [new_counts[i] for i in range(10)]}, index=CLASS_NAMES)
cmp.plot(kind='bar', figsize=(10,4), title='Training objects per class: before vs after'); plt.tight_layout(); plt.show()

In [ ]:
# Experiment 2: SAME settings as custom_baseline, only the dataset changes.
_ = train_and_eval('E2_dataset_aug', DATA2_YAML, epochs=30, imgsz=640, batch=16)
show_results('E2_dataset_aug')

✍️ **YOUR TURN (Experiment 2):** compare `E2_dataset_aug` with `custom_baseline` in the comparison table (section 9 or just run `pd.DataFrame(RESULTS).T`).
Did the weak classes improve? Did anything get worse? **Why do you think so?**

In [ ]:
# Which dataset is better so far? Experiments 3 and 4 continue from the better one.
BEST_DS = best_experiment(['custom_baseline', 'E2_dataset_aug'])
BEST_YAML = DATA2_YAML if BEST_DS == 'E2_dataset_aug' else DATA_YAML
print('Continuing with the dataset used by:', BEST_DS)

### Experiment 3: Training optimisation
Two **separate** single-change runs (so you can tell which one helped):
- **3a:** more epochs (30 → 60). Watch for overfitting in the curves.
- **3b:** larger image size (640 → 800). Objects are small, so more pixels might help; it will be slower.

In [ ]:
_ = train_and_eval('E3a_epochs60', BEST_YAML, epochs=60, imgsz=640, batch=16)
_ = train_and_eval('E3b_imgsz800',  BEST_YAML, epochs=30, imgsz=800, batch=8)   # batch 8: bigger images use more GPU memory
show_results('E3a_epochs60'); show_results('E3b_imgsz800')

✍️ **YOUR TURN (Experiment 3):** did more epochs or a bigger image size help? In `results.png`, does validation loss keep falling or start to rise (overfitting)?
What was the cost (see `train_min` and `infer_ms`)?

### Experiment 4: Advanced improvement (augmentation)
**Augmentation** = YOLO randomly changes images during training (flip, scale, mosaic, colour jitter...) so the model sees more variety.
- **4a: colour-safe augmentation:** `hsv_h=0` (no hue shift, because hue *is* the label), `hsv_s=0.3` (smaller saturation jitter).
- **4b: stronger geometric augmentation:** rotation ±10°, scale ±50%, light mixup, while keeping default colour jitter.

In [ ]:
_ = train_and_eval('E4a_color_safe_aug', BEST_YAML, epochs=30, imgsz=640, batch=16, hsv_h=0.0, hsv_s=0.3)
_ = train_and_eval('E4b_geometric_aug', BEST_YAML, epochs=30, imgsz=640, batch=16, degrees=10, scale=0.5, mixup=0.1)
show_results('E4a_color_safe_aug'); show_results('E4b_geometric_aug')

✍️ **YOUR TURN (Experiment 4):** which augmentation helped, and why? Did the colour-safe version reduce colour confusions
(e.g. red ↔ yellow/orange-ish) in the confusion matrix?

> **Tip:** results can vary a little between runs (random initialisation, augmentation). Differences of ~0.01 mAP are probably noise;
> don't over-claim. If you have time, re-run an experiment with a different `SEED` and see how much it moves.

## 8. Final inference demonstration

📘 **What & why**
Now we pick the **best model** (highest validation mAP50-95 among the fine-tuned runs) and test it on the **test** split, images it has **never** been trained or tuned on.
This is the number you report as the final performance. Run it **once**, it's your "exam".

In [ ]:
fine_tuned = [n for n in RESULTS if n != 'E1 pretrained']
FINAL_NAME = best_experiment(fine_tuned)
print('Final model:', FINAL_NAME, CONFIGS[FINAL_NAME])
final_model = YOLO(SAVE_DIRS[FINAL_NAME] / 'weights' / 'best.pt')

# Official test-set metrics (the required 5 metrics, per class + overall)
test_m = final_model.val(data=DATA_YAML, split='test', project=RUNS, name='final_test',
                         exist_ok=True, plots=True, verbose=False)
test_overall, test_per_class = summarize_metrics(test_m)
print('\nTEST SET, overall:'); print(pd.DataFrame([test_overall]).round(3).to_string(index=False))
print('\nTEST SET, per class:'); display(pd.DataFrame(test_per_class).T.round(3))

In [ ]:
# Visual demo on test images (many contain several objects). Left: ground truth. Right: model prediction.
demo = random.Random(1).sample(test_imgs, 4)
imgs, titles = [], []
for p in demo:
    imgs.append(draw_gt(p));                                   titles.append('Ground truth')
    r = final_model.predict(p, conf=0.25, verbose=False)[0]
    imgs.append(r.plot()[..., ::-1]);                          titles.append('Prediction')
show_grid(imgs, titles, cols=2, figsize=(12,4.2))

In [ ]:
# OPTIONAL: test on your OWN pictures (e.g. phone photos of coloured balls/rings).
# Upload some images and run this cell. Remember the model learned from synthetic images, so real photos may fail. That is a valuable finding to report!
# from google.colab import files; up = files.upload()
# for name in up:
#     r = final_model.predict(name, conf=0.25, verbose=False)[0]
#     plt.figure(figsize=(6,5)); plt.imshow(r.plot()[..., ::-1]); plt.axis('off'); plt.show()

## 8b. Classical colour analysis (HSV): the *non-learning* approach

📘 **What & why: this is the heart of the "ColorVision" assignment**

YOLO *learns* colour as part of a class name. But we can also **measure colour directly from pixels**:

1. **Preprocess:** convert **RGB → HSV**. HSV separates *what colour* (**H**ue, 0-179 in OpenCV) from *how colourful* (**S**aturation) and *how bright* (**V**alue).
   Lighting changes mostly affect V, so hue is far more reliable than raw RGB for naming colours.
2. **Isolate the object:** inside the object's bounding box there is still background, shadow, white highlights, white ring centre and black outline.
   We remove them with a **saturation/brightness mask**: only pixels with `S ≥ s_min` and `V ≥ v_min` are "real colour" (white/grey/black have low S or V).
3. **Extract & classify:** map each remaining pixel's hue to a colour name (hue ranges below).
4. **Multicolour analysis:** compute the **percentage** of each colour; colours above `min_frac` are *significant*.
   One significant colour → that colour. Two or more → **multicolor**.
5. **K-means alternative:** instead of fixed hue ranges, *cluster* the pixels into k groups automatically, then name each cluster's centre colour.

Hue wheel (OpenCV, 0-179): red wraps around (0-8 **and** 172-179), orange 8-20, yellow 20-35, green 35-85, cyan 85-100, blue 100-130, purple 130-155, pink 155-172.

In [ ]:
HUE_BINS = [('red', 0, 8), ('orange', 8, 20), ('yellow', 20, 35), ('green', 35, 85), ('cyan', 85, 100),
            ('blue', 100, 130), ('purple', 130, 155), ('pink', 155, 172), ('red', 172, 180)]

def hue_name(h):
    for name, lo, hi in HUE_BINS:
        if lo <= h < hi: return name
    return 'red'

def colour_percentages(crop_bgr, method='hsv', s_min=90, v_min=60, min_frac=0.08, k=4):
    # Return {colour_name: percent} for the SIGNIFICANT colours in an object crop (percentages add to 100).
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)                 # step 1: RGB(BGR) -> HSV
    mask = (hsv[..., 1] >= s_min) & (hsv[..., 2] >= v_min)          # step 2: keep only saturated, bright pixels
    if mask.sum() < 20: return {}
    counts = Counter()
    if method == 'hsv':                                              # step 3: fixed hue ranges
        for h in hsv[..., 0][mask]: counts[hue_name(h)] += 1
    elif method == 'kmeans':                                         # step 5 alternative: cluster in BGR space
        px = crop_bgr[mask].astype(np.float32)
        if len(px) < k * 5: return {}
        cv2.setRNGSeed(0)
        _, lab, centers = cv2.kmeans(px, k, None, (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0),
                                     3, cv2.KMEANS_PP_CENTERS)
        for i, c in enumerate(centers):
            ch = cv2.cvtColor(c.reshape(1, 1, 3).astype(np.uint8), cv2.COLOR_BGR2HSV)[0, 0, 0]
            counts[hue_name(int(ch))] += int((lab.ravel() == i).sum())
    else:
        raise ValueError(method)
    total = sum(counts.values())
    sig = {n: c for n, c in counts.items() if c / total >= min_frac}  # step 4: drop tiny (noise/edge) colours
    tot2 = sum(sig.values())
    return dict(sorted({n: 100 * c / tot2 for n, c in sig.items()}.items(), key=lambda x: -x[1]))

def colour_label(pct):
    # one significant colour -> its name; several -> 'multicolor'
    if not pct: return 'unknown'
    return list(pct)[0] if len(pct) == 1 else 'multicolor'

# --- Visualise the pipeline on one object so you can SEE each step ---
def show_pipeline(img_path, which=0, **kw):
    img = cv2.imread(img_path); h, w = img.shape[:2]
    c, x1, y1, x2, y2 = read_labels(label_path(img_path), w, h)[which]
    crop = img[y1:y2, x1:x2]
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    mask = (hsv[..., 1] >= kw.get('s_min', 90)) & (hsv[..., 2] >= kw.get('v_min', 60))
    pct = colour_percentages(crop, **kw)
    fig, ax = plt.subplots(1, 4, figsize=(14, 3.4))
    ax[0].imshow(crop[..., ::-1]); ax[0].set_title(f'1. crop ({CLASS_NAMES[c]})')
    ax[1].imshow(hsv[..., 0], cmap='hsv', vmin=0, vmax=179); ax[1].set_title('2. Hue channel')
    ax[2].imshow(mask, cmap='gray'); ax[2].set_title('3. colour mask (white = kept)')
    ax[3].bar(list(pct), list(pct.values()), color=[('tab:' + n if n in ('red','orange','green','blue','purple','pink','cyan') else n) for n in pct])
    ax[3].set_title(f'4. colours → {colour_label(pct)}'); ax[3].set_ylabel('%')
    for a in ax[:3]: a.axis('off')
    plt.tight_layout(); plt.show()

for p in random.Random(3).sample(train_imgs, 3): show_pipeline(p)

In [ ]:
# --- Full visualisation: original | detected regions + colour labels + percentages ---
def visualise_colours(img_path, boxes=None, model=None, method='hsv', **kw):
    # boxes: list of (x1,y1,x2,y2). If None and model is given, YOLO finds them. Otherwise ground truth boxes are used.
    img = cv2.imread(img_path); h, w = img.shape[:2]
    if boxes is None and model is not None:
        r = model.predict(img_path, conf=0.25, verbose=False)[0]
        boxes = [tuple(map(int, b)) for b in r.boxes.xyxy.tolist()]
    if boxes is None:
        boxes = [g[1:] for g in read_labels(label_path(img_path), w, h)]
    out = img.copy()
    for (x1, y1, x2, y2) in boxes:
        pct = colour_percentages(img[y1:y2, x1:x2], method=method, **kw)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 160, 0), 2)
        text = colour_label(pct) + ': ' + ', '.join(f'{n} {p:.0f}%' for n, p in pct.items())
        cv2.putText(out, text[:46], (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 0, 0), 3)
        cv2.putText(out, text[:46], (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].imshow(img[..., ::-1]); ax[0].set_title('Original'); ax[1].imshow(out[..., ::-1]); ax[1].set_title('Regions + colour labels + %')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

for p in random.Random(5).sample(test_imgs, 2):
    visualise_colours(p, model=final_model)       # YOLO finds the objects, HSV measures their colour (hybrid)

### Evaluating the colour methods against ground truth

We compare three ways to decide an object's **colour** (`red`, `blue`, `green`, `yellow`, or `multicolor`), using the test-set labels as the verified ground truth
(the colour part of the class name, e.g. `red_ring` → `red`):

| Approach | How |
|---|---|
| **A. YOLO class** | take the colour from YOLO's predicted class name (a missed object counts as wrong) |
| **B. HSV on ground-truth boxes** | HSV analysis on perfect boxes, which isolates how good the colour method itself is |
| **C. Hybrid (YOLO box + HSV)** | YOLO finds the object, HSV names the colour (the realistic end-to-end system) |

In [ ]:
def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0

def cache_yolo_preds(model, paths, conf=0.25):
    out = {}
    for p in paths:
        r = model.predict(p, conf=conf, verbose=False)[0]
        out[p] = [(int(c), *map(int, b)) for c, b in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist())]
    return out

def evaluate_colour(paths, yolo_preds, **cfg):
    rows = []
    for p in paths:
        img = cv2.imread(p); h, w = img.shape[:2]
        for g in read_labels(label_path(p), w, h):
            true_class = CLASS_NAMES[g[0]]; true_col = COLOR_OF(true_class)
            gt_box = g[1:]
            # B: HSV on the ground-truth box
            b_pred = colour_label(colour_percentages(img[g[2]:g[4], g[1]:g[3]], **cfg))
            # match the best YOLO prediction (IoU >= 0.5)
            best = max(yolo_preds[p], key=lambda q: iou(gt_box, q[1:]), default=None)
            if best is not None and iou(gt_box, best[1:]) >= 0.5:
                a_pred = COLOR_OF(CLASS_NAMES[best[0]])                                  # A
                c_pred = colour_label(colour_percentages(img[best[2]:best[4], best[1]:best[3]], **cfg))  # C
            else:
                a_pred = c_pred = 'MISSED'
            rows.append(dict(image=os.path.basename(p), true_class=true_class, true=true_col,
                             A_yolo=a_pred, B_hsv_gt=b_pred, C_hybrid=c_pred))
    return pd.DataFrame(rows)

def accuracy_table(df):
    acc = {col: (df[col] == df['true']).mean() for col in ['A_yolo', 'B_hsv_gt', 'C_hybrid']}
    return {k: round(v, 3) for k, v in acc.items()}

yolo_test_preds = cache_yolo_preds(final_model, test_imgs)
COLOUR_CFG = dict(method='hsv', s_min=90, v_min=60, min_frac=0.08)
col_df = evaluate_colour(test_imgs, yolo_test_preds, **COLOUR_CFG)
print('Colour accuracy on the test set (', len(col_df), 'objects ):', accuracy_table(col_df))
print('\nErrors:'); display(col_df[(col_df['A_yolo'] != col_df['true']) | (col_df['B_hsv_gt'] != col_df['true']) | (col_df['C_hybrid'] != col_df['true'])].head(20))

### Colour-method experiments (controlled, one change at a time)

| Config | Change | Question |
|---|---|---|
| **H0** | **no background removal** (`s_min=0, v_min=0`) | Why do we need a mask at all? |
| **H1** | baseline: `s_min=90, v_min=60, min_frac=0.08` | reference |
| **H2** | looser mask `s_min=60, v_min=40` | does letting in paler pixels help or hurt? |
| **H3** | stricter significance `min_frac=0.15` | does ignoring small colours reduce false "multicolor"? |
| **H4** | **K-means** (k=4) instead of fixed hue ranges | learned clustering vs hand-set ranges |

> Each row is evaluated on the **validation** images so we don't tune on the test set; the final choice is then checked on test.

In [ ]:
val_imgs = sorted(glob.glob(f'{DATA}/images/val/*.jpg'))
yolo_val_preds = cache_yolo_preds(final_model, val_imgs)

HSV_EXPERIMENTS = {
    'H0 no mask':         dict(method='hsv',    s_min=0,  v_min=0,  min_frac=0.08),
    'H1 baseline HSV':    dict(method='hsv',    s_min=90, v_min=60, min_frac=0.08),
    'H2 looser mask':     dict(method='hsv',    s_min=60, v_min=40, min_frac=0.08),
    'H3 min_frac 0.15':   dict(method='hsv',    s_min=90, v_min=60, min_frac=0.15),
    'H4 k-means (k=4)':   dict(method='kmeans', s_min=90, v_min=60, min_frac=0.08, k=4),
}
hsv_rows = {}
for name, cfg in HSV_EXPERIMENTS.items():
    d = evaluate_colour(val_imgs, yolo_val_preds, **cfg)
    hsv_rows[name] = accuracy_table(d)
hsv_table = pd.DataFrame(hsv_rows).T
display(hsv_table)
hsv_table[['B_hsv_gt', 'C_hybrid']].plot(kind='bar', figsize=(9, 3.5), ylim=(0, 1.05), title='Colour accuracy on validation images')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

In [ ]:
# Final colour result: best HSV config (by hybrid accuracy on VAL) checked once on TEST.
BEST_HSV = hsv_table['C_hybrid'].idxmax()
print('Best colour configuration on validation:', BEST_HSV, HSV_EXPERIMENTS[BEST_HSV])
final_col_df = evaluate_colour(test_imgs, yolo_test_preds, **HSV_EXPERIMENTS[BEST_HSV])
print('TEST colour accuracy:', accuracy_table(final_col_df))

# Per-class colour accuracy for the hybrid system
pc = final_col_df.assign(ok=final_col_df['C_hybrid'] == final_col_df['true']).groupby('true_class')['ok'].mean().round(3)
display(pc.to_frame('hybrid colour accuracy'))

✍️ **YOUR TURN (colour experiments):** explain (a) why H0 fails (what does the background / white ring centre contribute?), (b) which setting changed the results most and why,
(c) where HSV beats YOLO's class names or loses to them, and (d) which colours are hardest (hint: look at orange/yellow/red boundaries and the multicolour objects with orange, purple, cyan or pink parts).

## 9. Comparison and discussion

All model versions side by side (validation set), plus the final test result.

In [ ]:
summary = pd.DataFrame(RESULTS).T[['P', 'R', 'mAP50', 'mAP50_95', 'infer_ms', 'train_min']].round(3)
display(summary)

summary[['P', 'R', 'mAP50', 'mAP50_95']].plot(kind='bar', figsize=(11, 4), ylim=(0, 1.05), title='All experiments (validation set)')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

# Per-class mAP50-95 heat-map: reveals which classes each experiment helped or hurt
pcm = pd.DataFrame({n: {c: v['mAP50_95'] for c, v in d.items()} for n, d in PER_CLASS.items()})
plt.figure(figsize=(10, 5)); plt.imshow(pcm.values, cmap='viridis', vmin=0, vmax=1, aspect='auto'); plt.colorbar(label='mAP50-95')
plt.xticks(range(pcm.shape[1]), pcm.columns, rotation=30, ha='right'); plt.yticks(range(pcm.shape[0]), pcm.index)
for i in range(pcm.shape[0]):
    for j in range(pcm.shape[1]): plt.text(j, i, f'{pcm.values[i, j]:.2f}', ha='center', va='center', color='w', fontsize=8)
plt.title('Per-class mAP50-95 by experiment'); plt.tight_layout(); plt.show()

In [ ]:
# Look at the model's MISTAKES on the test set: where does it fail?
def error_cases(model, paths, conf=0.25):
    bad = []
    for p in paths:
        img = cv2.imread(p); h, w = img.shape[:2]
        gts = read_labels(label_path(p), w, h)
        r = model.predict(p, conf=conf, verbose=False)[0]
        preds = [(int(c), *map(int, b)) for c, b in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist())]
        for g in gts:
            m = max(preds, key=lambda q: iou(g[1:], q[1:]), default=None)
            if m is None or iou(g[1:], m[1:]) < 0.5:
                bad.append((p, 'missed ' + CLASS_NAMES[g[0]]))
            elif m[0] != g[0]:
                bad.append((p, f'{CLASS_NAMES[g[0]]} → predicted {CLASS_NAMES[m[0]]}'))
    return bad

errs = error_cases(final_model, test_imgs)
print(f'{len(errs)} errors on the test set')
if errs:
    sel = errs[:6]
    show_grid([final_model.predict(p, conf=0.25, verbose=False)[0].plot()[..., ::-1] for p, _ in sel], [e for _, e in sel], cols=3)

✍️ **YOUR TURN: discussion (write in full sentences, with numbers from the tables above)**
- **Strengths:** which classes/experiments worked best? Which model is fastest?
- **Weaknesses & errors:** what does the confusion matrix / error gallery show? Why might those mistakes happen?
- **YOLO vs HSV vs hybrid:** which gave the best *colour* accuracy and why? When would you use each?
- **Limits:** the data is synthetic with plain backgrounds, so what would break on real photos (lighting, shadows, textured backgrounds, partly hidden objects)?

## 10. Conclusion and references

✍️ **YOUR TURN:** write a short conclusion covering:
1. **What improved** (with numbers: baseline → final mAP50 / mAP50-95 / colour accuracy).
2. **What did not improve** (honest negative results are good science).
3. **Limitations** (synthetic dataset; small test set of 30 images; results may vary run to run).
4. **Future work** (add real labelled photos, more lighting variety, try a bigger model such as `yolo11s`).

**References** (edit as needed)
- Ultralytics YOLO documentation: https://docs.ultralytics.com
- Ultralytics performance metrics guide: https://docs.ultralytics.com/guides/yolo-performance-metrics/
- Ultralytics training tips / hyperparameters: https://docs.ultralytics.com/guides/model-training-tips/
- OpenCV colour spaces and `cvtColor`: https://docs.opencv.org/4.x/df/d9d/tutorial_py_colorspaces.html
- OpenCV K-means clustering: https://docs.opencv.org/4.x/d1/d5c/tutorial_py_kmeans_opencv.html
- Dataset: *ColorVision AI synthetic YOLO dataset (300 images)*, provided by the course instructor (procedurally generated).
- COCO dataset (pretrained weights): Lin et al., "Microsoft COCO: Common Objects in Context", 2014.

In [ ]:
# Save everything (results table, plots, best weights) to Google Drive so a Colab disconnect can't erase your work.
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ColorVision_AI_results'
os.makedirs(OUT, exist_ok=True)
pd.DataFrame(RESULTS).T.to_csv(f'{OUT}/experiment_results.csv')
hsv_table.to_csv(f'{OUT}/colour_experiments.csv')
shutil.copytree(RUNS, f'{OUT}/runs', dirs_exist_ok=True)
print('Saved to', OUT)